# SPICE — Phase 2 Post-train / RL (Colab)

> Two paths: Path A (SAC micro) + Path B (ES macro) + quick screening / phase maps / pseudo-label reflow, backed by `spice_engine`.
>
> **Prerequisite**: `spice_engine` is compiled into a **whl**; this notebook installs it in step 2 (if Colab has no engine, you can skip step 5 and demo the pure-TF components instead).
>
> Run order: ① deps + engine → ② code → ③ load Pre-train artifacts → ④ pure-TF component demo → ⑤ run two-path RL → ⑥ fine-tune / visualize.

In [ ]:
# ① Dependencies (China pip mirror) + HF mirror + GPU
!pip install -q -i https://pypi.tuna.tsinghua.edu.cn/simple tensorflow datasets huggingface_hub pyarrow polars pyyaml

import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

## ② Install spice_engine whl

`spice_engine` is a PyO3 library compiled from the local `md_cal` (Linux whl). **Choose one**:
- Option A: upload the `.whl` file to the Colab root directory
- Option B: put a direct whl URL in the code below

In [ ]:
import glob, subprocess, sys

# Option A: whl uploaded to /content
wheels = glob.glob("/content/*.whl")
if wheels:
    print("Installing:", wheels[0])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", wheels[0]])

# Option B: direct URL (uncomment and fill in the address)
# url = "https://your-host/spice_engine-0.1.0-cp312-cp312-manylinux_2_17_x86_64.whl"
# subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", url])

import spice_engine
print("spice_engine OK, version:", spice_engine.version())

## ③ Get the project code

Put `model/` under Google Drive's `MyDrive/SPICE/model` (recommended) or package it as `model.zip` and upload.

In [ ]:
import os, sys, zipfile

try:
    from google.colab import drive
    drive.mount("/content/drive")
    proj = "/content/drive/MyDrive/SPICE/model"
    if os.path.isdir(proj):
        os.chdir(proj); print("Using Drive project:", proj)
except Exception as e:
    print("Skipping Drive:", e)

if not os.path.isdir("spice_rl"):
    for zp in ("/content/model.zip", "/content/SPICE.zip"):
        if os.path.exists(zp):
            with zipfile.ZipFile(zp) as z:
                z.extractall("/content")
            if os.path.isdir("/content/model"):
                os.chdir("/content/model")
            break

sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd(), "| spice_pre:", os.path.isdir("spice_pre"), "| spice_rl:", os.path.isdir("spice_rl"))

## ④ How RL consumes Pre-train artifacts

The three artifacts produced by Pre-train (`spice_pre`) are all reused by RL:

| Pre-train artifact | Purpose | Consumed in RL by |
|---|---|---|
| `checkpoints/pretrain/best_weights.weights.h5` | backbone + Head A weights | `train_post.build_rl_model` loads with `skip_mismatch=True` (Head B/C/D are random, left to ES/SAC) |
| `data/tfrecords/shard_*.tfrecord` | pre-training data | `finetune_pretrain` merges with the original data when reflowing |
| `configs/pretrain.yaml` | model-structure hyperparameters (embed_dim, etc.) | `build_rl_model` reads the model structure to stay consistent with Pre-train |

These correspond to the `pretrain_ckpt` / `pretrain_tfrecord_dir` / `pretrain_config` pointers in `configs/posttrain.yaml`. The cell below verifies loading:

In [ ]:
from spice_pre.config import load_config as pre_load
from spice_pre.models import SPICEPretrainModel
from spice_rl.config import load_config as rl_load

rl_cfg = rl_load("configs/posttrain.yaml")
pre_cfg = pre_load(rl_cfg.post.pretrain_config)   # read the Pre-train model structure

# Build the full two-path model (all heads enabled)
import numpy as np, tensorflow as tf
model = SPICEPretrainModel(pre_cfg.model, heads=("A", "B", "Bp", "C", "D"))
model({"tokens": tf.zeros([1,8], tf.int32), "env": tf.zeros([1,3]), "mask": tf.ones([1,8])}, training=False)

import os
if os.path.exists(rl_cfg.post.pretrain_ckpt):
    model.load_weights(rl_cfg.post.pretrain_ckpt, skip_mismatch=True)
    print("Loaded Pre-train weights (backbone + Head A)")
print("All five heads present:", all(getattr(model, h) is not None for h in
      ("head_a","head_b","head_bp","head_c","head_d")))
print("embed_dim:", model.embed_dim)

## ⑤ Pure-TF component demo (runs without the engine)

SAC dual-head action sampling / ES policy & mutation / Head D confidence.

In [ ]:
from spice_rl.sac import SACTrainer
import numpy as np

cont_dim = rl_cfg.env.force_dim + rl_cfg.env.env_offset_dim
sac = SACTrainer(rl_cfg.sac, z_dim=model.embed_dim, cont_dim=cont_dim,
                 u_window=rl_cfg.env.u_window)
print("SAC target entropy:", round(sac.target_entropy, 3), "(= -(18+2)×0.5)")

L = rl_cfg.sac.discrete_position_dim
z_mask = np.concatenate([np.ones(40, np.float32), np.zeros(L-40, np.float32)])
a_cont, a_disc = sac.act(np.random.randn(model.embed_dim).astype(np.float32),
                         np.random.rand(3).astype(np.float32), z_mask)
print("Continuous action (bias force 16 + ΔpH,ΔT):", a_cont.shape)
print("Discrete mutation (pos one-hot + AA):", a_disc.shape)

In [ ]:
from spice_rl.es import ESEvolver
from spice_pre.data.preprocessing import seq_to_tokens

es = ESEvolver(model, rl_cfg.es)
print("ES evolvable variables (Head-B/C + policy vector):", len(es.head_vars))

seq = "MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQ"   # example sequence
tokens = seq_to_tokens(seq)
mask = np.ones(len(tokens), np.float32)
env = np.array([0.5, 0.5, 0.5], np.float32)
cands = es.propose_mutations(seq, tokens, env, mask)
print("Top 5 mutation candidates (policy conservative/aggressive):")
for s, k, st in cands[:5]:
    print("  ", s[:40], "...", "mut=", k, st)

In [ ]:
# Head D: two-path confidence training (pure TF, supervised label = survival steps / max steps)
from spice_rl.confidence import ConfidenceHeadTrainer

ct = ConfidenceHeadTrainer(model, lr=rl_cfg.post.conf_lr)
for _ in range(64):
    ct.add(np.random.randn(model.embed_dim).astype(np.float32),
           np.array([0.5, 0.8], np.float32))
print("Head D conf loss:", round(float(ct.update(32)["conf_loss"]), 4))
print("Head D prediction:", np.round(ct.predict(np.random.randn(model.embed_dim).astype(np.float32)), 3))

## ⑥ Run two-path RL (requires the engine)

`train_post` main loop: quick screening → Path A (SAC dual-thread perturbation explores boundaries + phase maps) → Path B (ES mutation, frozen SAC-Actor evaluation, survivors → pseudo-labels).

Parameters live in `configs/posttrain.yaml` (overridable with `--max-episodes` below). Pass an initial structure as mmCIF, e.g. `md_cal/data/test/2LYZ.cif` (upload it to /content and fill in the path).

In [ ]:
# Two-path RL (debug: few episodes; for production increase max_episodes)
rl_cfg.env.relax_iters = 100
rl_cfg.post.max_episodes = 5

import os, shutil
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

# Load the structure from the data pipeline (heavy atoms → from_atoms; the engine adds hydrogens automatically, no mmCIF needed)
from huggingface_hub import hf_hub_download
os.makedirs("/content/parquet", exist_ok=True)
p = hf_hub_download("SPICE-Protein/spice_protein", "atoms_shard_0001.parquet",
                    repo_type="dataset")
shutil.copy(p, "/content/parquet/atoms_shard_0001.parquet")

import polars as pl
pdb_ids = pl.read_parquet("/content/parquet/atoms_shard_0001.parquet",
                          columns=["pdb_id"])["pdb_id"].unique().to_list()
print("pdb_ids available in this shard (first 10):", pdb_ids[:10])
PDB_ID = pdb_ids[0]

from spice_rl.env import load_structure_with_atoms
struct, base_atoms = load_structure_with_atoms("/content/parquet", PDB_ID)
seq = struct.sequence()
print("Initial structure:", seq[:30], "... n_res =", struct.residue_count(), "| pdb_id =", PDB_ID)

from spice_rl.train_post import train
train(rl_cfg, struct, seq, base_atoms=base_atoms)

## ⑦ Pseudo-label reflow → fine-tune (can run without the engine)

Time-averaged coordinates of surviving Path-B mutants (`data/pseudo_labels/pseudo_*.npz`) are reflowed with confidence weighting, merged with the original Pre-train TFRecords to fine-tune Head A → `finetuned.weights.h5`.

In [ ]:
from spice_rl.finetune_pretrain import finetune
rl_cfg.post.finetune_epochs = 1
finetune(rl_cfg)

## ⑧ Visualization: stability phase maps

Reads `runs/posttrain/phase_maps/*.npz` (the pH-T plane scanned by the engine) and plots stable/collapse boundaries. If there's no data yet, an example is shown to demonstrate the axes.

In [ ]:
import glob, numpy as np
import matplotlib.pyplot as plt

files = sorted(glob.glob("runs/posttrain/phase_maps/*.npz"))
if files:
    d = np.load(files[-1], allow_pickle=True)
    ph, temp, stable = d["ph"], d["temp"], d["stable"]
    plt.figure(figsize=(7,5))
    plt.scatter(ph, temp, c=stable.astype(int), cmap="RdYlGn", s=60,
                vmin=0, vmax=1)
    plt.colorbar(label="stable (1=stable, 0=collapsed)")
    plt.xlabel("pH"); plt.ylabel("Temperature (K)")
    plt.title(f"Stability phase map {files[-1].split('/')[-1]}")
    plt.show()
else:
    # No data: example phase map
    ph = np.linspace(0, 14, 29); T = np.linspace(280, 340, 13)
    PH, TT = np.meshgrid(ph, T)
    stable = (TT < 320) & (PH > 3) & (PH < 11)
    plt.figure(figsize=(7,5))
    plt.pcolormesh(PH, TT, stable.astype(int), cmap="RdYlGn", shading="auto")
    plt.colorbar(label="stable")
    plt.xlabel("pH"); plt.ylabel("Temperature (K)"); plt.title("Stability phase map (example)")
    plt.show()
    print("(Example data: running train_post generates real phase maps that will be shown automatically)")

## Summary

- Pre-train artifacts (weights / TFRecords / structure config) are reused by RL via the three pointers in `configs/posttrain.yaml`.
- Two-path RL: quick screening → Path A SAC explores boundaries + phase maps → Path B ES mutation (frozen SAC-Actor evaluation) → pseudo-label reflow & fine-tune.
- Outputs: `runs/posttrain/phase_maps/*.npz` (phase maps), `data/pseudo_labels/*.npz` (pseudo-labels), `checkpoints/pretrain/finetuned.weights.h5` (fine-tuned weights).